# 🌌 AstroLoc-ML · End-to-End Walkthrough

One notebook covering the full project: data, renderer, model, loss, training
result, classical comparison, and honest interpretation.

**What you'll see:**
1. The HYG star catalog and how we sample synthetic pointings.
2. Gnomonic projection and the renderer producing star-field images.
3. Why MSE on raw RA/Dec is wrong and what we use instead.
4. The 7-output `(sin, cos, log_scale)` model architecture.
5. Training history loaded from the trained checkpoint.
6. The classical triangle-hash solver (the baseline that actually works).
7. Honest interpretation: where the ML model fails and what would fix it.

**Prerequisites:** activate the venv, install Jupyter, run from the project root.
```bash
source .venv/bin/activate
pip install jupyter ipykernel
jupyter notebook notebooks/astroloc_ml_demo.ipynb
```

## 0 · Setup

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
%cd {ROOT}

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

# Pick whatever GPU we have
if torch.cuda.is_available(): DEVICE = 'cuda'
elif torch.backends.mps.is_available(): DEVICE = 'mps'
else: DEVICE = 'cpu'
print(f'torch={torch.__version__}  device={DEVICE}')

## 1 · The data: HYG star catalog

**HYG v3** is a free amateur-astronomy catalog of ~120,000 stars with right
ascension, declination, and apparent magnitude. We keep stars with `mag <= 8`
(visible to a consumer camera) — about 41,000 stars.

Why magnitude matters: stars are inverse-log brightness, so a Pogson's-law
rescaling (`100^(-mag/5)`) gives us their relative photon flux for rendering.

In [ ]:
from src.data.catalog import load_hyg_catalog
catalog = load_hyg_catalog('data/catalogs/hygdata_v3.csv', mag_limit=8.0)
print(f'Loaded {len(catalog):,} stars with mag <= 8.0')
catalog.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(catalog['mag'], bins=60, color='#7c5cff', edgecolor='white', linewidth=0.4)
axes[0].set_xlabel('Visual magnitude'); axes[0].set_ylabel('Count')
axes[0].set_title('Magnitude distribution (mag ≤ 8)')

ax = axes[1]; ax.remove()
ax = fig.add_subplot(1, 2, 2, projection='mollweide')
ra = np.deg2rad(catalog['ra_deg'].to_numpy() - 180.0)
dec = np.deg2rad(catalog['dec_deg'].to_numpy())
size = np.clip(8.0 - catalog['mag'], 0.2, 6.0)
ax.scatter(ra, dec, s=size, c='black', alpha=0.5, edgecolors='none')
ax.grid(True, alpha=0.3); ax.set_title('Sky coverage (Mollweide)')
plt.show()

**Insight:** the magnitude distribution is heavy in the dim end (most stars are barely
visible). The sky coverage map shows the Milky Way as a bright band — this matters
because random uniform sampling on the sphere will pick galactic-plane fields more often
than equatorial-pole fields, giving us natural class balance for star density.

## 2 · Synthetic data pipeline

We render unlimited training images by:
1. Sampling a random pointing **uniformly on the sphere** (not uniform in RA/Dec — that
   would over-represent the poles).
2. Filtering the catalog to stars within the field of view.
3. **Gnomonic-projecting** sky → tangent plane.
4. Rotating the field by a random angle, then scaling to pixels.
5. Splatting each star as a magnitude-weighted Gaussian PSF.
6. Adding Poisson photon noise + Gaussian readout noise + optional sky gradient.

**Gnomonic** is the right projection because it preserves great circles as straight lines
through the tangent point — the correct geometry for small-FOV astrophotography.

In [ ]:
from src.utils.coordinates import gnomonic_project

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, dec0 in zip(axes, [0.0, 45.0, 80.0]):
    ras, decs = np.meshgrid(np.arange(-30, 31, 5), np.arange(-30, 31, 5))
    xs, ys = gnomonic_project(ras + 180.0, decs + dec0, 180.0, dec0)
    ax.plot(xs, ys, color='#7c5cff', linewidth=0.8)
    ax.plot(xs.T, ys.T, color='#7c5cff', linewidth=0.8)
    ax.set_title(f'Tangent plane @ Dec={dec0:.0f}°'); ax.set_aspect('equal')
fig.suptitle('Gnomonic projection warps more aggressively near the pole')
plt.tight_layout(); plt.show()

In [ ]:
from src.data.renderer import render_star_field, sample_random_pointing

fig, axes = plt.subplots(1, 4, figsize=(14, 4), facecolor='black')
for ax, fw in zip(axes, [15, 30, 50, 80]):
    img = render_star_field(ra_center=85, dec_center=-2, field_width=fw, rotation=0,
                            catalog=catalog, image_size=224,
                            rng=np.random.default_rng(int(fw)))
    ax.imshow(img); ax.set_title(f'FOV {fw}°', color='white'); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('Synthetic star fields at increasing field width', color='white')
plt.tight_layout(); plt.show()

**Insight:** at 15° FOV you can resolve individual bright stars; at 80° you see large
swaths of the sky with most stars unresolved. The renderer pulls the right catalog subset
for whatever FOV the network is being trained on — this is what makes synthetic data
scalable: one renderer, infinite samples, every sky position covered.

## 3 · Augmentations — the night sky has no canonical orientation

Standard image-classification augmentations (`RandomHorizontalFlip` + small rotations)
would teach the network an orientation prior the data doesn't have. The night sky is
rotationally symmetric. We use **full 180° random rotations + flips** on every sample.

In [ ]:
from PIL import Image
from src.data.augmentations import build_train_transforms
tfm = build_train_transforms(224)
base = render_star_field(85, -2, 30, 0, catalog, image_size=224, rng=np.random.default_rng(7))
base_pil = Image.fromarray((base * 255).astype(np.uint8))
fig, axes = plt.subplots(2, 4, figsize=(12, 6), facecolor='black')
for ax in axes.flatten():
    t = tfm(base_pil)
    img = (t.permute(1, 2, 0).numpy() * np.array([0.15, 0.15, 0.20])
           + np.array([0.10, 0.10, 0.15]))
    ax.imshow(np.clip(img, 0, 1)); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('Same input, eight different augmented samples (180° rotation + flips + jitter + blur)',
             color='white')
plt.tight_layout(); plt.show()

## 4 · The loss function story — why MSE on raw RA/Dec is wrong

**Right ascension wraps at 0°/360°.** If the truth is RA=1° and the model predicts
RA=359°, the *real* angular error is 2° (going the short way around the sphere), but
naive MSE sees `(359-1)^2 = 128,164` and panics. So you'd think: use the great-circle
(haversine) distance as the loss. That's wrap-invariant.

But there's a second problem we discovered the hard way:

In [ ]:
from src.utils.coordinates import angular_separation_deg
ra_truth = 1.0
ra_preds = np.linspace(0, 360, 361)
mse = (ra_preds - ra_truth) ** 2
ang = angular_separation_deg(ra_preds, 0, ra_truth, 0)
fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
ax1.plot(ra_preds, mse, color='#f56565', label='MSE')
ax2.plot(ra_preds, ang, color='#3ad29f', label='Great-circle Δ')
ax1.set_xlabel('Predicted RA (°)'); ax1.set_ylabel('MSE', color='#f56565')
ax2.set_ylabel('Great-circle Δ (°)', color='#3ad29f')
ax1.set_title('Truth RA=1°. MSE explodes at the wrap; great-circle stays continuous.')
plt.tight_layout(); plt.show()

**The second problem:** even with haversine loss, the model has no incentive to keep its
predictions in valid ranges (`[0, 360)` for RA, `[-90, +90]` for Dec). Because the loss is
*wrap-invariant*, predicting `RA = -89°` is locally equivalent to predicting `RA = +271°`.
The optimizer wanders in unbounded space and converges glacially.

**Our actual fix:** predict `(sin RA, cos RA, sin Dec, cos Dec, sin rot, cos rot,
log_scale)` — 7 outputs — and reconstruct angles via `atan2`. The mapping from any
angle to `(sin, cos)` is smooth and bounded; `atan2` gives back the angle in the right
range automatically. The loss landscape becomes smooth everywhere, no wrap discontinuity,
no out-of-range predictions possible.

In [ ]:
from src.models.astrolocnet import AstroLocNet
from src.training.loss import angular_separation_loss

# Sanity check the loss does what we claim
pred = torch.tensor([[1.0, 0.0,  0.0, 1.0,  0.0, 1.0,  np.log(30.0)]])    # decodes to RA=90, Dec=0, rot=0
target = torch.tensor([[90.0, 0.0, 0.0, np.log(30.0)]])
print(f'Perfect prediction:  loss = {angular_separation_loss(pred, target).item():.6f}')

pred = torch.tensor([[0.0, 1.0,  0.0, 1.0,  0.0, 1.0,  np.log(30.0)]])    # decodes to RA=0
print(f'Off by 90° in RA:    loss = {angular_separation_loss(pred, target).item():.6f}')

## 5 · Model architecture

EfficientNet-B0 backbone (ImageNet pretrained, ~4 M parameters) + a small regression
head producing the 7 sin/cos/log_scale outputs.

In [ ]:
from src.models.astrolocnet import AstroLocNet, freeze_backbone, unfreeze_last_n_blocks

model = AstroLocNet(pretrained=False)
total = sum(p.numel() for p in model.parameters())
head  = sum(p.numel() for p in model.head_parameters)
backbone = total - head
print(f'Total params:    {total:>10,}')
print(f'Backbone:        {backbone:>10,} ({backbone/total:.1%})')
print(f'Regression head: {head:>10,} ({head/total:.1%})')
print()
x = torch.randn(2, 3, 224, 224)
print(f'Forward: in={tuple(x.shape)} → out={tuple(model(x).shape)}  (B, [sin_ra, cos_ra, ...])')

In [ ]:
# Freezing strategy across the three training phases.
snapshots = {}
snapshots['Phase 0 — everything trainable'] = sum(p.numel() for p in model.parameters() if p.requires_grad)
freeze_backbone(model)
snapshots['Phase 1 — head only']           = sum(p.numel() for p in model.parameters() if p.requires_grad)
unfreeze_last_n_blocks(model, n=3)
snapshots['Phase 2 — last 3 blocks + head'] = sum(p.numel() for p in model.parameters() if p.requires_grad)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(list(snapshots.keys()), [c/1e6 for c in snapshots.values()], color='#7c5cff')
ax.set_xlabel('Trainable params (millions)'); ax.set_title('Trainable parameters per training phase')
for i, c in enumerate(snapshots.values()):
    ax.text(c/1e6, i, f'  {c:,}', va='center')
plt.tight_layout(); plt.show()

**Insight:** Phase 1 trains just the regression head against frozen ImageNet features —
fast, cheap, gives the head a reasonable initialization. Phase 2 then unfreezes the last
three EfficientNet blocks (high-level abstractions) at a much lower learning rate so the
backbone can specialize toward star fields without destroying its general features.

## 6 · Training: what actually happened

We load the saved checkpoint and inspect the per-epoch history. To re-train end-to-end:
```bash
python train.py --config configs/default.yaml --device mps --skip-phase3 \
  --train-samples 20000 --val-samples 1000 --num-workers 6 \
  --epochs-phase1 5 --epochs-phase2 10
```
(`caffeinate -i` on macOS to prevent the laptop from sleeping mid-run.)

In [ ]:
ckpt_path = ROOT / 'checkpoints' / 'best.pt'
if not ckpt_path.exists():
    print('No checkpoint yet — run `python train.py --smoke --skip-phase3 --device mps` first.')
else:
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    print(f'best val ang sep: {ckpt["best_metric"]:.3f}°')
    rows = [{'phase': h['phase'], 'epoch': h['epoch'],
             'train_loss': h['train']['loss'], 'val_loss': h['val']['loss'],
             'val_sep_deg': h['val']['ang_sep_mean_deg'],
             'val_median_deg': h['val']['ang_sep_median_deg'],
             'within_5°_%': h['val']['pct_within_5_deg'],
             'within_1°_%': h['val']['pct_within_1_deg']} for h in ckpt['history']]
    df = pd.DataFrame(rows).round(3)
    df

In [ ]:
if ckpt_path.exists():
    df['x'] = range(len(df))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(df['x'], df['train_loss'], label='train', marker='o', color='#7c5cff')
    axes[0].plot(df['x'], df['val_loss'], label='val', marker='s', color='#ffd166')
    axes[0].set_title('Loss'); axes[0].set_xlabel('epoch idx'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(df['x'], df['val_sep_deg'], color='#3ad29f', marker='o')
    axes[1].axhline(90, color='gray', linestyle=':', label='random baseline (90°)')
    axes[1].set_title('Val angular separation (°) — lower is better')
    axes[1].set_xlabel('epoch idx'); axes[1].legend(); axes[1].grid(alpha=0.3)
    axes[2].plot(df['x'], df['within_5°_%'], color='#f5a524', marker='o', label='within 5°')
    axes[2].plot(df['x'], df['within_1°_%'], color='#f56565', marker='s', label='within 1°')
    axes[2].set_title('Tolerance hit rates (%)'); axes[2].set_xlabel('epoch idx')
    axes[2].legend(); axes[2].grid(alpha=0.3)
    for ax in axes:
        for i in range(1, len(df)):
            if df['phase'].iloc[i] != df['phase'].iloc[i-1]:
                ax.axvline(i - 0.5, color='gray', linestyle='--', alpha=0.4)
    fig.tight_layout(); plt.show()

**Honest interpretation:** loss descends smoothly (the sin/cos parameterization is working —
no wrap-around discontinuity), but **val angular separation barely moves**. The model is
learning to satisfy the `(sin, cos)` unit-norm constraint but is *not* learning the actual
image → sky-coordinate mapping. Within-5° hit rate stays under 1%.

This is a clean illustration of three real-world ML lessons:

1. **Pretrained ImageNet features don't transfer to star fields.** EfficientNet learned to
   spot edges, textures, and object parts — none of which exist in a star field. The
   frozen backbone in Phase 1 is essentially random feature extraction for our task.
2. **End-to-end deep regression is the wrong tool for geometric matching.** Identifying
   "these three bright stars in this pattern = Orion's belt" is what asterism matching
   does well, and what off-the-shelf CNNs do poorly without orders of magnitude more data.
3. **20K samples + 5 M parameters is in the regime where the model memorizes statistics
   rather than learning underlying geometry.** Pre-rendering 100K+ samples and training for
   50+ epochs would help; reframing as classification over HEALPix tiles would help more.

## 7 · The classical solver — the baseline that actually works

`src/models/classical_solver.py` implements the asterism-matching algorithm that
Astrometry.net uses (at much larger scale):

1. CLAHE + SimpleBlobDetector → detect star centroids in the image.
2. Build an invariant **triangle hash table** from catalog stars: every triple gets a
   rotation-/scale-/translation-invariant signature (the sorted side-length ratios).
3. At query time: form triangles from detected centroids, look up the hash, let candidate
   matches vote on the field-center catalog star.

The same triangle, rotated and scaled, produces the same invariant:

In [ ]:
from src.models.classical_solver import triangle_invariant
tri = np.array([[0, 0], [3, 0], [1, 2]], dtype=float)
theta = np.deg2rad(37); R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
tri_rotscaled = (tri @ R.T) * 2.7 + np.array([5, -3])
print('Original:        ', triangle_invariant(*tri))
print('Rotated+scaled+shifted:', triangle_invariant(*tri_rotscaled))

In [ ]:
from src.models.classical_solver import ClassicalSolver
solver = ClassicalSolver(catalog, max_catalog_stars=300, max_triangle_arc_deg=20.0)
print('Building triangle hash index over the brightest 300 catalog stars...')
solver.build_index(verbose=True)

In [ ]:
img = render_star_field(85.0, -2.0, 25.0, 0.0, catalog, image_size=224, rng=np.random.default_rng(0))
img_u8 = (img * 255).astype(np.uint8)
result = solver.solve(img_u8)
print(f'Truth: RA=85.0°  Dec=-2.0°')
print(f'Classical solve: RA={result.ra:.1f}° Dec={result.dec:.1f}° votes={result.n_matches} t={result.solve_time*1000:.0f}ms')

fig, ax = plt.subplots(figsize=(6, 6), facecolor='black')
ax.imshow(img); ax.set_facecolor('black'); ax.set_xticks([]); ax.set_yticks([])
if len(result.centroids):
    ax.scatter(result.centroids[:, 0], result.centroids[:, 1],
               facecolors='none', edgecolors='#7c5cff', s=80, linewidths=1.2,
               label=f'{len(result.centroids)} detected stars')
ax.legend(loc='lower right', facecolor='black', edgecolor='white', labelcolor='white')
ax.set_title(f'Classical solve: RA={result.ra:.1f}°  Dec={result.dec:.1f}°  (votes={result.n_matches})',
             color='white')
plt.tight_layout(); plt.show()

## 8 · Side-by-side: ML vs classical on the same image

Renders a synthetic field at a known pointing, then runs both solvers on the same input.
Bigger picture: every research project that frames a problem as "replace classical X with
neural network Y" should run an ablation like this one.

In [ ]:
from src.inference.predict import load_model, predict_image

model_trained = load_model('checkpoints/best.pt', device='cpu') if (ROOT/'checkpoints/best.pt').exists() else None
rng = np.random.default_rng(2026)
rows = []
for i in range(5):
    ra, dec, rot, fw = sample_random_pointing(rng, (20.0, 60.0))
    img = render_star_field(ra, dec, fw, rot, catalog, image_size=224, rng=np.random.default_rng(i))
    img_u8 = (img * 255).astype(np.uint8)

    if model_trained is not None:
        ml_pred = predict_image(model_trained, img_u8)
        ml_sep = angular_separation_deg(ra, dec, ml_pred.ra_deg, ml_pred.dec_deg)
    else:
        ml_sep = float('nan')

    cl = solver.solve(img_u8)
    cl_sep = angular_separation_deg(ra, dec, cl.ra, cl.dec) if cl.success else float('nan')

    rows.append({'truth_ra': round(ra,1), 'truth_dec': round(dec,1),
                 'ml_sep_deg': round(float(ml_sep), 2),
                 'classical_sep_deg': round(float(cl_sep), 2),
                 'classical_votes': cl.n_matches})
pd.DataFrame(rows)

**Insight:** the classical solver's per-sample error depends on whether its triangle
voting finds a unique winner; when it does, it's typically within a few degrees of truth.
The ML model's errors are large (tens of degrees) and largely uncorrelated with the truth.
This is the empirical justification for keeping the classical solver as the production
path and treating the ML model as a research baseline.

## 9 · Visual: model prediction vs truth

Render an image, run the trained model, overlay catalog stars at the *predicted* WCS.
If the model were correct, the purple circles would land on the bright dots in the image.
Spoiler: they don't.

In [ ]:
if model_trained is None:
    print('No trained model yet — skip.')
else:
    from src.inference.visualize import plot_constellation_overlay
    rng = np.random.default_rng(11)
    for case in range(3):
        ra, dec, rot, fw = sample_random_pointing(rng, (25.0, 55.0))
        img = render_star_field(ra, dec, fw, rot, catalog, image_size=224, rng=np.random.default_rng(case))
        img_u8 = (img * 255).astype(np.uint8)
        pred = predict_image(model_trained, img_u8)
        fig = plot_constellation_overlay(
            img_u8,
            ra_center=pred.ra_deg, dec_center=pred.dec_deg,
            field_width_deg=pred.field_width_deg, rotation_deg=pred.rotation_deg,
            catalog=catalog, mag_limit=5.5,
            title=f'Pred (RA, Dec)=({pred.ra_deg:.1f}°, {pred.dec_deg:.1f}°)  '
                  f'vs truth ({ra:.1f}, {dec:.1f})',
        )
        plt.show()

## 10 · Takeaways

**What this project demonstrates well:**
- Principled synthetic-data generation from a real star catalog.
- Gnomonic projection — the correct geometry for small fields.
- `(sin, cos)` parameterization — the correct fix for spherical regression.
- Three-phase fine-tuning with differential learning rates.
- Honest ablation against a working classical baseline.
- Honest evaluation that doesn't paper over the negative result.

**What didn't work, and why:**
- The ML model converges to ~76° val angular separation (random baseline is ~90°).
  It's barely learning the actual task.
- Root cause: domain mismatch between ImageNet pretraining and star fields, combined
  with insufficient data (20K samples) and the wrong inductive bias for the problem
  (end-to-end regression of a fundamentally geometric task).

**What would fix it (future work):**
- Pre-render 100K+ samples to disk and train 50+ epochs (removes the CPU-bound renderer
  bottleneck, lets the network see far more data per wall-clock minute).
- Reframe as classification over HEALPix tiles instead of regression over the sphere.
- Add Phase 3 fine-tuning on real Astrometry.net solved images (closes the
  synthetic→real domain gap).
- Try a smaller from-scratch CNN (~1 M params) instead of an ImageNet backbone.

**The bottom line:** the classical triangle-hash solver (`src/models/classical_solver.py`)
is the practical tool. The ML side of this project is valuable as a teaching artifact:
a fully wired pipeline with an honest negative result is more useful than a polished one
with fudged numbers.